# Создание и заполнение данных БД Postgre

In [48]:
%pip install python-dotenv psycopg2-binary

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import os
import json
import psycopg2
from psycopg2.extras import DictCursor
from dotenv import load_dotenv


# Получение секретов

In [51]:
# получаем текущую директорию ноутбука 
current_dir = os.getcwd()

# переходим на один уровень вверх
project_root = os.path.dirname(current_dir)

# формируем путь к файлу .env в папке Task1, там у нас лежит файл .env с настройками подключения к БД
dotenv_path = os.path.join(project_root, 'task_2_Docker', '.env')

# загружаем переменные окружения из указанного файла
load_dotenv(dotenv_path)

# получим доступ к переменным окружения
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
db_name = os.getenv("DB_NAME")
db_port = os.getenv("DB_PORT") 
secret_hash = os.getenv("SECRET_HASH") 

print(f"Загруженные данные: USER={user}, DB={db_name}, DB_PORT={db_port}")

Загруженные данные: USER=schepetova, DB=my_db_schepetova, DB_PORT=5433


# Подключение к базе данных PostgreSQL

In [52]:
conn = None
try:
    conn = psycopg2.connect(
        host="localhost", # если Docker контейнер запущен локально, а ноутбук вне Docker.
                          # НО! если ноутбук также в Docker и в одной сети с БД,
                          # то нужно использовать имя сервиса Docker (например, 'db' или 'postgres_db').
        database=db_name,
        user=user,
        password=password,
        port=db_port
    )
    cursor = conn.cursor()

    print("Успешное подключение к базе данных!")
    
except Exception as e:
    print(f"Ошибка при подключении к базе данных: {e}")

Успешное подключение к базе данных!


In [53]:
# пример запроса
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print(f"Версия PostgreSQL: {db_version}")

Версия PostgreSQL: ('PostgreSQL 13.23 (Debian 13.23-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [54]:
# получить список таблиц:
cursor.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
""")
tables = cursor.fetchall()
print("\nТаблицы в базе данных:")
for table in tables:
    print(f"- {table[0]}")


Таблицы в базе данных:
- departments
- user_logs


In [55]:
# закрытие соединения с БД - После завершения работы с БД не забываем закрывать соединение!
# cursor.close()
# conn.close()

Вам предоставлена БД с логами (действиями) студентов на образовательном портале за весенний семестр (агрегация по каждой неделе) по отдельному электронному курсу - таблица user_logs (примечание. создана в предыдущих л.р.).
- сourseid — уникальный идентификатор курса, дисциплины;
- userid — уникальный идентификатор студента (не используется в обучении);
- num_week — номер недели в году;
- s_all — количество всех событий на текущий момент;
- s_all_avg — среднее количество всех событий в неделю;
- s_course_viewed — количество просмотров курса;
- s_course_viewed_avg — среднее количество просмотров курса в неделю;
- s_q_attempt_viewed — количество просмотров теста;
- s_q_attempt_viewed_avg — среднее количество просмотров теста в неделю;
- s_a_course_module_viewed — количество просмотров модуля в курсе;
- s_a_course_module_viewed_avg — среднее количество просмотров модуля в курсе в неделю;
- s_a_submission_status_viewed — количество отправленных заданий на проверку;
- s_a_submission_status_viewed_avg — среднее количество ответов;
- namer_level — оценка за дисциплину;
- depart — номер кафедры;
- name_osno — основа обучения (имеет два значения: бюджет или контракт);
- name_formopril — форма обучения;
- leveled — уровень образования (имеет два значения: бакалавриат, магистратура);
- num_sem — номер семестра;
- kurs — номер курса учебной группы.

Также в таблице  departments хранятся названия кафедр, таблица связана с логами по полю depart:
id - код кафедры;
name - сокращенное название кафедры. 

## Задание 1 (если до этого еще этот шаг не был выполнен):

Измените данные вещественного типа, сейчас целая и дробная часть разделены запятой, замените ее на точку. 

Выведите первые 10 записей, чтобы проверить результат предобработки. 

In [56]:
cursor.execute("SELECT * FROM user_logs LIMIT 10")
rows = cursor.fetchall()

print("\nПервые 10 записей после предобработки:")
for row in rows:
    print(row)


Первые 10 записей после предобработки:
(71262, 34527, 6, 9, 9.0, 4, 4.0, 0, 0.0, 0, 0.0, 0, 0.0, 3, 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34609, 6, 6, 6.0, 3, 3.0, 0, 0.0, 0, 0.0, 0, 0.0, 2, 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34610, 6, 13, 13.0, 5, 5.0, 0, 0.0, 1, 1.0, 1, 1.0, 5, 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34611, 6, 12, 12.0, 7, 7.0, 0, 0.0, 0, 0.0, 0, 0.0, 4, 'Экзамен', 22, '2', '1', '1', 2, 2, '18.06.2022')
(71262, 34612, 6, 24, 24.0, 8, 8.0, 0, 0.0, 0, 0.0, 0, 0.0, 3, 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34613, 6, 15, 15.0, 8, 8.0, 0, 0.0, 0, 0.0, 0, 0.0, 4, 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34615, 6, 14, 14.0, 7, 7.0, 0, 0.0, 0, 0.0, 0, 0.0, 4, 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34616, 6, 5, 5.0, 2, 2.0, 0, 0.0, 0, 0.0, 0, 0.0, 4, 'Экзамен', 22, '1', '1', '1', 2, 2, '18.06.2022')
(71262, 34616, 8, 2, 3.0, 2, 2.0, 0, 0.0, 0, 0.0, 0, 0.0, 4, '

## Задание 2: 

Выведите количество кафедр, за которыми закреплены курсы на портале.





In [57]:
cursor.execute("""
    SELECT COUNT(DISTINCT depart) as departments_count
    FROM user_logs
    WHERE depart IS NOT NULL
""")
result = cursor.fetchone()
print(f"\nКоличество кафедр, за которыми закреплены курсы: {result[0]}")


Количество кафедр, за которыми закреплены курсы: 43


##  Задание 3:

Выведите сколько у каждой кафедры закреплено электронных курсов на портале. 
Требуется выводит сокращенное название кафедры и количество курсов. 
У какой кафедры больше всего курсов на портале?

In [58]:
cursor.execute("""
    SELECT 
        d.name as department_name,
        COUNT(DISTINCT u.courseid) as courses_count
    FROM user_logs u
    JOIN departments d ON u.depart = d.id
    WHERE u.depart IS NOT NULL
    GROUP BY d.name
    ORDER BY courses_count DESC
""")
results = cursor.fetchall()

print("\nКоличество курсов по кафедрам:")
print("-" * 50)
for dept_name, count in results:
    print(f"{dept_name}: {count} курсов")

# Кафедра с наибольшим количеством курсов
max_courses = max(results, key=lambda x: x[1])
print(f"\nКафедра с наибольшим количеством курсов: {max_courses[0]} ({max_courses[1]} курсов)")


Количество курсов по кафедрам:
--------------------------------------------------
ДиСО: 53 курсов
ПОиД: 42 курсов
МиХТ: 42 курсов
ГМиТТК: 40 курсов
ЛиУТС: 36 курсов
БИиИТ: 35 курсов
ГМУиУП: 35 курсов
ПиЭММО: 33 курсов
АЭПиМ: 33 курсов
ЛиП: 33 курсов
ПиСЗ: 32 курсов
Эконом.: 32 курсов
ТОМ: 30 курсов
ГМДиОПИ: 29 курсов
ЛПиМ: 28 курсов
РМПИ: 28 курсов
МиТОДиМ: 28 курсов
ВТиП: 25 курсов
Психол.: 23 курсов
ВИ: 22 курсов
ТССА: 21 курсов
Дизайна: 20 курсов
Менеджм.: 20 курсов
ЯиЛ: 20 курсов
CC: 19 курсов
ПМиИ: 19 курсов
АиИИ: 19 курсов
ТиЭС: 19 курсов
Химии: 18 курсов
ХОМ: 18 курсов
РЯОЯиМК: 17 курсов
ЭПП: 16 курсов
ИиИБ: 16 курсов
АСУ: 16 курсов
Физики: 16 курсов
ЭиМЭ: 16 курсов
УиИС: 15 курсов
СРиППО: 14 курсов
ПЭиБЖД: 11 курсов
Физкульт.: 5 курсов
ЦДОМ: 4 курсов
ИТМ: 3 курсов
УСиБА: 2 курсов

Кафедра с наибольшим количеством курсов: ДиСО (53 курсов)


## Задание 4:

Ответьте на вопрос: существуют ли курсы, за которыми закреплено несколько кафедр? Если такие курсы есть, то выведите их количество.
Также выведите названия кафедр, которые совместно преподают один и тот же курс.




In [59]:
# Находим курсы, которые принадлежат нескольким кафедрам
cursor.execute("""
    WITH course_departments AS (
        SELECT 
            courseid,
            COUNT(DISTINCT depart) as dept_count,
            STRING_AGG(DISTINCT d.name, ', ') as departments
        FROM user_logs u
        JOIN departments d ON u.depart = d.id
        GROUP BY courseid
    )
    SELECT 
        COUNT(*) as courses_with_multiple_depts,
        courseid,
        departments
    FROM course_departments
    WHERE dept_count > 1
    GROUP BY courseid, departments
""")
results = cursor.fetchall()

if results:
    print(f"\nКоличество курсов, закрепленных за несколькими кафедрами: {len(results)}")
    print("\nЭти курсы и кафедры:")
    for course_id, dept_count, departments in results:
        print(f"Курс {course_id}: {departments}")
else:
    print("\nКурсов, закрепленных за несколькими кафедрами, не найдено")


Количество курсов, закрепленных за несколькими кафедрами: 60

Эти курсы и кафедры:
Курс 1: ГМиТТК, ПиСЗ, УиИС
Курс 1: ПиСЗ, УиИС
Курс 1: ПиСЗ, УиИС
Курс 1: ПиСЗ, УиИС
Курс 1: ГМиТТК, ТиЭС
Курс 1: ЛиУТС, ПиСЗ, УиИС
Курс 1: CC, ВТиП
Курс 1: ГМДиОПИ, ЭиМЭ
Курс 1: АЭПиМ, ЭПП
Курс 1: АЭПиМ, ЭПП
Курс 1: АЭПиМ, ЭПП
Курс 1: АЭПиМ, ТиЭС, ЭПП
Курс 1: АЭПиМ, ЭПП
Курс 1: АЭПиМ, УиИС
Курс 1: Дизайна, ЛПиМ
Курс 1: Дизайна, ЛПиМ
Курс 1: МиХТ, ТОМ
Курс 1: ЛПиМ, МиХТ, ТОМ
Курс 1: ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 1: ЛПиМ, МиХТ, ТОМ
Курс 1: ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 1: ЛПиМ, МиХТ, ТОМ
Курс 1: ЛПиМ, МиХТ, ТОМ
Курс 1: ГМДиОПИ, ЛПиМ, МиХТ, ТОМ
Курс 1: ЛПиМ, МиХТ, ТОМ
Курс 1: МиХТ, ТОМ
Курс 1: МиХТ, ТОМ
Курс 1: АЭПиМ, ХОМ
Курс 1: Дизайна, ТОМ
Курс 1: Дизайна, ТОМ, ХОМ
Курс 1: CC, ДиСО
Курс 1: БИиИТ, Эконом.
Курс 1: ГМДиОПИ, ГМиТТК, РМПИ
Курс 1: ГМДиОПИ, ГМиТТК, ЛиУТС, РМПИ
Курс 1: ГМДиОПИ, ГМиТТК, РМПИ
Курс 1: ГМДиОПИ, ГМиТТК, РМПИ
Курс 1: ГМДиОПИ, ГМиТТК, РМПИ
Курс 1: ПиСЗ, Эконом.
Курс 1: АСУ, АЭПиМ, ВИ,

## Задание 5:

Выведите количество студентов, которые получили 2, 3, 4, 5.

Пример вывода:

| namer_level |	count |
|-----|------|
|2 |	4 |
|3 |	3435 |
|4 | 	4676765|
|5 | 232 |


In [60]:
cursor.execute("""
    SELECT 
        namer_level as grade,
        COUNT(DISTINCT userid) as student_count
    FROM user_logs
    WHERE namer_level IN (2, 3, 4, 5)
    GROUP BY namer_level
    ORDER BY namer_level
""")
results = cursor.fetchall()

print("\nКоличество студентов по оценкам:")
print("-" * 30)
print(f"{'Оценка':<10} {'Количество студентов':<20}")
print("-" * 30)
for grade, count in results:
    print(f"{grade:<10} {count:<20}")


Количество студентов по оценкам:
------------------------------
Оценка     Количество студентов
------------------------------
2          1069                
3          1884                
4          3243                
5          3407                


## Задание 6:

Выведите студента, который больше всех работает на портале (у него максимальное количество логов за вест период обучения).

In [61]:
cursor.execute("""
    SELECT 
        userid,
        SUM(s_all) as total_logs
    FROM user_logs
    GROUP BY userid
    ORDER BY total_logs DESC
    LIMIT 1
""")
result = cursor.fetchone()
print(f"\nСтудент с максимальным количеством логов: userid={result[0]}, всего логов={result[1]}")


Студент с максимальным количеством логов: userid=28871, всего логов=10300


## Задание 7:

Выведите по каждой недели среднее количество всех событий на портале.

In [62]:
cursor.execute("""
    SELECT 
        num_week,
        AVG(s_all) as avg_events
    FROM user_logs
    GROUP BY num_week
    ORDER BY num_week
""")
results = cursor.fetchall()

print("\nСреднее количество событий по неделям:")
print("-" * 40)
print(f"{'Неделя':<10} {'Среднее количество событий':<25}")
print("-" * 40)
for week, avg_events in results:
    print(f"{week:<10} {avg_events:.2f}")


Среднее количество событий по неделям:
----------------------------------------
Неделя     Среднее количество событий
----------------------------------------
6          13.80
7          9.62
8          8.03
9          9.39
10         8.21
11         10.02
12         9.38
13         10.01
14         9.86
15         10.35
16         10.29
17         10.52
18         9.67
19         11.11
20         14.45
21         18.50
22         22.49
23         22.26
24         23.01
25         18.22
26         8.60
27         1.25
28         0.09
29         0.05


## Задание 8: 

Выведите название кафедры, у которой больше всего отличников.

Отдельно выведите название кафедры, у которой больше всего двоечников. 

In [63]:
# Кафедра с наибольшим количеством отличников (оценка 5)
cursor.execute("""
    SELECT 
        d.name as department_name,
        COUNT(DISTINCT u.userid) as excellent_students
    FROM user_logs u
    JOIN departments d ON u.depart = d.id
    WHERE u.namer_level = 5
    GROUP BY d.name
    ORDER BY excellent_students DESC
    LIMIT 1
""")
result_excellent = cursor.fetchone()

# Кафедра с наибольшим количеством двоечников (оценка 2)
cursor.execute("""
    SELECT 
        d.name as department_name,
        COUNT(DISTINCT u.userid) as poor_students
    FROM user_logs u
    JOIN departments d ON u.depart = d.id
    WHERE u.namer_level = 2
    GROUP BY d.name
    ORDER BY poor_students DESC
    LIMIT 1
""")
result_poor = cursor.fetchone()

print(f"\nКафедра с наибольшим количеством отличников: {result_excellent[0]} ({result_excellent[1]} студентов)")
print(f"Кафедра с наибольшим количеством двоечников: {result_poor[0]} ({result_poor[1]} студентов)")


Кафедра с наибольшим количеством отличников: ДиСО (310 студентов)
Кафедра с наибольшим количеством двоечников: Эконом. (72 студентов)


## Задание 9:
Провести анализ пиковой активности студентов перед экзаменом (с использованием (Common Table Expression — CTE), оператор with).

Вывести, на какой неделе семестра студенты проявляли наибольшую активность в курсе в целом, и как эта активность распределяется между студентами-бюджетниками и контрактниками.

Пример вывода :

| name_osno | week_number	| avg_s_all	| avg_s_course_viewed |	avg_s_q_attempt_viewed |
|-----|------|------|------|------|
| бюджет |	14	| 125.45 |	45.67 |	32.12 |
|контракт |	14	| 98.76 |	38.90 |	25.43 |

In [70]:
cursor.execute("""
WITH weekly_activity AS (
    SELECT 
        num_week,
        name_osno,
        AVG(s_all_avg) AS avg_s_all,
        AVG(s_course_viewed_avg) AS avg_s_course_viewed,
        AVG(s_q_attempt_viewed_avg) AS avg_s_q_attempt_viewed
    FROM user_logs
    GROUP BY num_week, name_osno
),
max_week AS (
    SELECT 
        name_osno,
        num_week,
        avg_s_all,
        avg_s_course_viewed,
        avg_s_q_attempt_viewed,
        ROW_NUMBER() OVER (PARTITION BY name_osno ORDER BY avg_s_all DESC) AS rn
    FROM weekly_activity
)
SELECT 
    name_osno,
    num_week AS week_number,
    ROUND(avg_s_all::numeric, 2) AS avg_s_all,
    ROUND(avg_s_course_viewed::numeric, 2) AS avg_s_course_viewed,
    ROUND(avg_s_q_attempt_viewed::numeric, 2) AS avg_s_q_attempt_viewed
FROM max_week
WHERE rn = 1
ORDER BY name_osno;
""")

rows = cursor.fetchall()

# Создаем таблицу
print(f"{'name_osno':<12} {'week_number':<14} {'avg_s_all':<12} {'avg_s_course_viewed':<20} {'avg_s_q_attempt_viewed':<15}")
print("-" * 80)

for row in rows:
    print(f"{row[0]:<12} {row[1]:<14} {row[2]:<12} {row[3]:<20} {row[4]:<15}")

name_osno    week_number    avg_s_all    avg_s_course_viewed  avg_s_q_attempt_viewed
--------------------------------------------------------------------------------
1            6              16.25        5.35                 1.00           
2            25             11.43        2.18                 1.69           
